# Example 5 - Exact h-transform with a variance penalty (two-blob collapse)

This notebook implements a **collective, unlabeled conditioning example** for the finite-dimensional Dirichlet-Ferguson particle system.

It is designed to match the requested constraints.

The terminal weight is

$$
g(\mu) = \exp\bigl(-\lambda\operatorname{Var}(\mu)\bigr),
$$

where $a$ is taken from the initial weighted variance.

The backward heat equation is solved exactly for this choice of $g$. The drift is obtained from the paper's recipe

$$
b_i^{(n)}(t,x;s) = 2 D_i^{(n)} \log u_t^{(n)}(x).
$$

No final particle locations are prescribed, so this is not an Algorithm 1 labelled-target example. The main visualization uses 128 particles, and the notebook includes a seed sweep and a size sweep to show that the effect is stable.

## Visual idea

Start from two separated blobs with the same weighted barycenter. Under the free diffusion the blobs broaden and remain spread out. Under the variance-penalized h-transform, every particle is pulled toward the current weighted barycenter, so the two blobs collapse into a single tight cloud.

That difference is visually clear even with $n = 128$ particles.


## Exact heat solve for the variance terminal datum

Freeze the masses $s = (s_1,\dots,s_n)$ and write the particle generator as

$$
L^{(n)} = \sum_{i=1}^n s_i^{-1}\Delta_{x_i}.
$$

For $x = (x_1,\dots,x_n) \in (\mathbb{R}^d)^n$, define the weighted barycenter by

$$
m(x) = \sum_{i=1}^n s_i x_i,
$$

and the weighted variance by

$$
V(x) = \sum_{i=1}^n s_i \|x_i - m(x)\|^2.
$$

We choose the terminal functional

$$
g_{\lambda,a}(x) = \exp\bigl(-\lambda(V(x)-a)\bigr) = e^{\lambda a} e^{-\lambda V(x)}.
$$

A useful fact is that the constant $a$ does not affect the drift: it only multiplies $u_t$ by the constant factor $e^{\lambda a}$, and

$$
D \log\bigl(e^{\lambda a} u_t\bigr) = D \log u_t.
$$

So the bias is controlled by $\lambda$, while $a$ only rescales the martingale density.

The key identities are

$$
\nabla_{x_i} V(x) = 2 s_i (x_i - m(x)),
$$

$$
\sum_{i=1}^n s_i^{-1} \|\nabla_{x_i} V(x)\|^2 = 4 V(x),
$$

and

$$
L^{(n)} V(x) = 2 d (n-1).
$$

Now solve the backward heat equation

$$
\partial_t u_t + L^{(n)} u_t = 0,
\qquad
u_T = g_{\lambda,a}.
$$

With $\tau = T-t$ and the ansatz

$$
u_t(x) = C(\tau) \exp\bigl(-\beta(\tau) V(x)\bigr),
$$

one gets

$$
\beta(\tau) = \frac{\lambda}{1 + 4 \lambda \tau},
\qquad
C(\tau) = e^{\lambda a} (1 + 4 \lambda \tau)^{-d(n-1)/2}.
$$

Hence the exact heat solution is

$$
u_t(x) = e^{\lambda a} (1 + 4 \lambda (T-t))^{-d(n-1)/2}
\exp\left(
-\frac{\lambda}{1 + 4 \lambda (T-t)} V(x)
\right).
$$

The corresponding h-transform drift is

$$
b_i(t,x) = \frac{2}{s_i} \nabla_{x_i} \log u_t(x)
= -\frac{4 \lambda}{1 + 4 \lambda (T-t)} (x_i - m(x)).
$$

So this is a collective mean-field contraction drift: each particle is pulled toward the current barycenter, not toward a prescribed endpoint.

### Why this is a clean example

- It uses the requested variance penalty.
- The heat solve is exact, not heuristic.
- The drift comes directly from $2 D \log u_t$, exactly as in the paper's h-transform framework.
- The weighted barycenter is unaffected by the drift.

Indeed,

$$
\sum_{i=1}^n s_i \, b_i(t,x) = 0.
$$

If the free and conditioned systems use the same Brownian increments, their barycenters coincide pathwise; only the spread changes.


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from dataclasses import dataclass
from pathlib import Path
from plotly.subplots import make_subplots
from IPython.display import display
import sys


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.support import circle_trace, configure_plotly
from core.wasserstein_conditioning_algorithms import ParticleSimulation

configure_plotly()
np.set_printoptions(precision=4, suppress=True)


In [2]:

def weighted_mean(points, masses):
    points = np.asarray(points, dtype=float)
    masses = np.asarray(masses, dtype=float).reshape(-1)
    return np.sum(masses[:, None] * points, axis=0)

def weighted_variance(points, masses):
    points = np.asarray(points, dtype=float)
    masses = np.asarray(masses, dtype=float).reshape(-1)
    mean = weighted_mean(points, masses)
    return float(np.sum(masses * np.sum((points - mean) ** 2, axis=1)))

def variance_htransform_u(points, masses, lambda_, a, t, horizon):
    points = np.asarray(points, dtype=float)
    tau = horizon - t
    n, d = points.shape
    var = weighted_variance(points, masses)
    prefactor = np.exp(lambda_ * a) * (1.0 + 4.0 * lambda_ * tau) ** (-0.5 * d * (n - 1))
    return float(prefactor * np.exp(-lambda_ * var / (1.0 + 4.0 * lambda_ * tau)))

def sunflower_blob(num_points, center, radius, angle_offset=0.0):
    idx = np.arange(1, num_points + 1, dtype=float)
    r = radius * np.sqrt((idx - 0.5) / num_points)
    theta = idx * (np.pi * (3.0 - np.sqrt(5.0))) + angle_offset
    return np.column_stack([
        center[0] + r * np.cos(theta),
        center[1] + r * np.sin(theta),
    ])

def make_two_blob_configuration(
    n_particles,
    left_center=(0.34, 0.55),
    right_center=(0.66, 0.45),
    radius=0.08,
):
    n_left = n_particles // 2
    n_right = n_particles - n_left
    left = sunflower_blob(n_left, left_center, radius, angle_offset=0.0)
    right = sunflower_blob(n_right, right_center, radius, angle_offset=np.pi / 7.0)
    points = np.vstack([left, right])
    blob_id = np.concatenate([
        np.zeros(n_left, dtype=float),
        np.ones(n_right, dtype=float),
    ])
    return points, blob_id, np.array([left_center, right_center], dtype=float), radius

@dataclass
class CoupledVarianceSimulation:
    times: np.ndarray
    masses: np.ndarray
    free: ParticleSimulation
    conditioned: ParticleSimulation
    weighted_means_free: np.ndarray
    weighted_means_conditioned: np.ndarray
    variances_free: np.ndarray
    variances_conditioned: np.ndarray
    lambda_: float
    a: float

def simulate_variance_htransform_pair(
    initial_positions,
    masses,
    lambda_,
    a,
    horizon,
    step_size,
    seed=0,
):
    x0 = np.asarray(initial_positions, dtype=float)
    masses = np.asarray(masses, dtype=float).reshape(-1)

    if x0.ndim != 2:
        raise ValueError("initial_positions must have shape (n, d)")
    if len(masses) != len(x0):
        raise ValueError("masses and initial_positions have incompatible sizes")
    if np.any(masses <= 0.0):
        raise ValueError("masses must be strictly positive")
    if not np.isclose(masses.sum(), 1.0):
        raise ValueError("masses must sum to 1")

    m_steps = int(round(horizon / step_size))
    if m_steps <= 0 or not np.isclose(m_steps * step_size, horizon):
        raise ValueError("horizon / step_size must be a positive integer")

    times = np.linspace(0.0, horizon, m_steps + 1)
    n, d = x0.shape
    rng = np.random.default_rng(seed)
    noise_scale = np.sqrt(2.0 * step_size / masses)[:, None]

    free_positions = np.empty((m_steps + 1, n, d), dtype=float)
    conditioned_positions = np.empty_like(free_positions)
    free_positions[0] = x0
    conditioned_positions[0] = x0

    free_drifts = np.zeros((m_steps, n, d), dtype=float)
    conditioned_drifts = np.empty((m_steps, n, d), dtype=float)

    free_means = np.empty((m_steps + 1, d), dtype=float)
    conditioned_means = np.empty_like(free_means)
    free_variances = np.empty(m_steps + 1, dtype=float)
    conditioned_variances = np.empty_like(free_variances)

    free_means[0] = weighted_mean(x0, masses)
    conditioned_means[0] = free_means[0]
    free_variances[0] = weighted_variance(x0, masses)
    conditioned_variances[0] = free_variances[0]

    for m in range(m_steps):
        noise = noise_scale * rng.normal(size=(n, d))

        # Free system
        free_positions[m + 1] = free_positions[m] + noise

        # Conditioned system with exact h-transform drift
        tau = horizon - times[m]
        mean_now = weighted_mean(conditioned_positions[m], masses)
        beta = lambda_ / (1.0 + 4.0 * lambda_ * tau)
        drift = -4.0 * beta * (conditioned_positions[m] - mean_now)

        conditioned_positions[m + 1] = conditioned_positions[m] + drift * step_size + noise
        conditioned_drifts[m] = drift

        free_means[m + 1] = weighted_mean(free_positions[m + 1], masses)
        conditioned_means[m + 1] = weighted_mean(conditioned_positions[m + 1], masses)
        free_variances[m + 1] = weighted_variance(free_positions[m + 1], masses)
        conditioned_variances[m + 1] = weighted_variance(conditioned_positions[m + 1], masses)

    free_sim = ParticleSimulation(
        times=times,
        positions=free_positions,
        masses=masses,
        drifts=free_drifts,
        lifted_positions=free_positions.copy(),
    )
    conditioned_sim = ParticleSimulation(
        times=times,
        positions=conditioned_positions,
        masses=masses,
        drifts=conditioned_drifts,
        lifted_positions=conditioned_positions.copy(),
    )

    return CoupledVarianceSimulation(
        times=times,
        masses=masses,
        free=free_sim,
        conditioned=conditioned_sim,
        weighted_means_free=free_means,
        weighted_means_conditioned=conditioned_means,
        variances_free=free_variances,
        variances_conditioned=conditioned_variances,
        lambda_=float(lambda_),
        a=float(a),
    )


In [3]:
BLOB_COLORSCALE = [
    [0.0, "royalblue"],
    [0.499, "royalblue"],
    [0.501, "crimson"],
    [1.0, "crimson"],
]

def particle_trace(points, color_values, name, marker_size=9, showlegend=False):
    return go.Scatter(
        x=points[:, 0],
        y=points[:, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=marker_size,
            color=color_values,
            colorscale=BLOB_COLORSCALE,
            cmin=0.0,
            cmax=1.0,
            showscale=False,
            line=dict(color="black", width=0.6),
            opacity=0.92,
        ),
        showlegend=showlegend,
        text=[f"particle {i}" for i in range(len(points))],
        hovertemplate="%{text}<br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
    )

def center_marker(point, name="weighted barycenter", color="black", size=11, showlegend=False):
    return go.Scatter(
        x=[point[0]],
        y=[point[1]],
        mode="markers",
        name=name,
        marker=dict(symbol="x", size=size, color=color, line=dict(color=color, width=1)),
        showlegend=showlegend,
        hovertemplate="barycenter<br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
    )

def make_snapshot_grid(sim, color_values, snapshot_indices, x_range=(0.0, 1.0), y_range=(0.0, 1.0)):
    times = [sim.times[idx] for idx in snapshot_indices]
    fig = make_subplots(
        rows=2,
        cols=len(snapshot_indices),
        subplot_titles=[
            *(f"free — t = {t:.6f}" for t in times),
            *(f"conditioned — t = {t:.6f}" for t in times),
        ],
        horizontal_spacing=0.03,
        vertical_spacing=0.10,
    )

    for col, idx in enumerate(snapshot_indices, start=1):
        fig.add_trace(particle_trace(sim.free.positions[idx], color_values, "free", marker_size=8), row=1, col=col)
        fig.add_trace(center_marker(sim.weighted_means_free[idx]), row=1, col=col)
        fig.add_trace(particle_trace(sim.conditioned.positions[idx], color_values, "conditioned", marker_size=8), row=2, col=col)
        fig.add_trace(center_marker(sim.weighted_means_conditioned[idx]), row=2, col=col)

    fig.update_layout(
        width=1240,
        height=700,
        template="simple_white",
        title="Free diffusion vs variance-conditioned h-transform — snapshot strip",
        margin=dict(t=90, l=40, r=20, b=40),
    )

    for row in [1, 2]:
        for col in range(1, len(snapshot_indices) + 1):
            fig.update_xaxes(range=list(x_range), title="x", row=row, col=col)
            fig.update_yaxes(range=list(y_range), title="y", row=row, col=col)

    return fig

def make_comparison_animation(
    sim,
    color_values,
    blob_centers=None,
    blob_radius=None,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
    title="Variance-penalized h-transform vs free diffusion",
):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Free diffusion", "Variance-conditioned h-transform"),
        horizontal_spacing=0.08,
    )

    if blob_centers is not None and blob_radius is not None:
        for col in [1, 2]:
            for j, center in enumerate(blob_centers):
                fig.add_trace(
                    circle_trace(
                        center=center,
                        radius=blob_radius,
                        name=f"initial blob {j + 1}",
                        showlegend=(col == 1),
                    ),
                    row=1,
                    col=col,
                )

    fig.add_trace(
        particle_trace(sim.free.positions[0], color_values, "free particles", marker_size=9),
        row=1,
        col=1,
    )
    fig.add_trace(
        center_marker(sim.weighted_means_free[0], showlegend=True),
        row=1,
        col=1,
    )
    fig.add_trace(
        particle_trace(sim.conditioned.positions[0], color_values, "conditioned particles", marker_size=9),
        row=1,
        col=2,
    )
    fig.add_trace(
        center_marker(sim.weighted_means_conditioned[0], showlegend=False),
        row=1,
        col=2,
    )

    free_particle_idx = len(fig.data) - 4
    free_center_idx = len(fig.data) - 3
    cond_particle_idx = len(fig.data) - 2
    cond_center_idx = len(fig.data) - 1

    frames = []
    for k, t in enumerate(sim.times):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(x=sim.free.positions[k][:, 0], y=sim.free.positions[k][:, 1]),
                    go.Scatter(x=[sim.weighted_means_free[k, 0]], y=[sim.weighted_means_free[k, 1]]),
                    go.Scatter(x=sim.conditioned.positions[k][:, 0], y=sim.conditioned.positions[k][:, 1]),
                    go.Scatter(x=[sim.weighted_means_conditioned[k, 0]], y=[sim.weighted_means_conditioned[k, 1]]),
                ],
                traces=[free_particle_idx, free_center_idx, cond_particle_idx, cond_center_idx],
                name=str(k),
                layout=go.Layout(title_text=f"{title}<br><sup>time = {t:.6f}</sup>"),
            )
        )
    fig.frames = frames

    slider_steps = [
        dict(
            method="animate",
            args=[[str(k)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
            label=f"{t:.6f}",
        )
        for k, t in enumerate(sim.times)
    ]

    fig.update_layout(
        title=f"{title}<br><sup>time = {sim.times[0]:.6f}</sup>",
        width=1120,
        height=620,
        template="simple_white",
        legend=dict(x=0.01, y=1.10, orientation="h"),
        sliders=[
            dict(
                active=0,
                currentvalue={"prefix": "time = "},
                pad={"t": 18},
                steps=slider_steps,
                x=0.12,
                y=-0.08,
                len=0.76,
            )
        ],
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                showactive=False,
                x=0.12,
                y=1.16,
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[None, {"frame": {"duration": 90, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True}],
                    ),
                    dict(
                        label="Pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
        margin=dict(t=90, l=40, r=20, b=60),
    )

    fig.update_xaxes(range=list(x_range), title="x", row=1, col=1)
    fig.update_yaxes(range=list(y_range), title="y", row=1, col=1)
    fig.update_xaxes(range=list(x_range), title="x", row=1, col=2)
    fig.update_yaxes(range=list(y_range), title="y", row=1, col=2)

    return fig

def make_variance_path_figure(sim):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=sim.times,
            y=sim.variances_free,
            mode="lines",
            name="free diffusion",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=sim.times,
            y=sim.variances_conditioned,
            mode="lines",
            name="variance-conditioned h-transform",
        )
    )
    fig.update_layout(
        title="Weighted variance over time",
        template="simple_white",
        width=860,
        height=420,
        xaxis_title="time",
        yaxis_title="weighted variance",
    )
    return fig

def seed_sweep_dataframe(n_particles, lambda_, horizon, step_size, seeds):
    masses = np.ones(n_particles, dtype=float) / n_particles
    initial_positions, _, _, _ = make_two_blob_configuration(n_particles)
    a_value = weighted_variance(initial_positions, masses)
    rows = []
    for seed in seeds:
        sim = simulate_variance_htransform_pair(
            initial_positions=initial_positions,
            masses=masses,
            lambda_=lambda_,
            a=a_value,
            horizon=horizon,
            step_size=step_size,
            seed=seed,
        )
        rows.append({
            "seed": int(seed),
            "terminal_variance_free": sim.variances_free[-1],
            "terminal_variance_conditioned": sim.variances_conditioned[-1],
            "reduction_factor": sim.variances_free[-1] / sim.variances_conditioned[-1],
        })
    return pd.DataFrame(rows)

def size_sweep_dataframe(n_values, lambda_, horizon, step_size, seeds):
    rows = []
    for n_particles in n_values:
        masses = np.ones(n_particles, dtype=float) / n_particles
        initial_positions, _, _, _ = make_two_blob_configuration(n_particles)
        a_value = weighted_variance(initial_positions, masses)

        free_vals = []
        cond_vals = []
        factors = []

        for seed in seeds:
            sim = simulate_variance_htransform_pair(
                initial_positions=initial_positions,
                masses=masses,
                lambda_=lambda_,
                a=a_value,
                horizon=horizon,
                step_size=step_size,
                seed=seed,
            )
            free_vals.append(sim.variances_free[-1])
            cond_vals.append(sim.variances_conditioned[-1])
            factors.append(sim.variances_free[-1] / sim.variances_conditioned[-1])

        rows.append({
            "n_particles": int(n_particles),
            "mean_terminal_variance_free": float(np.mean(free_vals)),
            "mean_terminal_variance_conditioned": float(np.mean(cond_vals)),
            "worst_case_reduction_factor": float(np.min(factors)),
        })

    return pd.DataFrame(rows)


In [4]:

# Main many-particle example
n_particles = 128
lambda_ = 20_000.0
horizon = 5.0e-5
n_steps = 240
step_size = horizon / n_steps
seed = 4

masses = np.ones(n_particles, dtype=float) / n_particles
initial_positions, blob_id, blob_centers, blob_radius = make_two_blob_configuration(
    n_particles=n_particles,
    left_center=(0.34, 0.55),
    right_center=(0.66, 0.45),
    radius=0.08,
)

# Match the requested functional g(mu)=exp(-lambda * (Var(mu)-a))
# Here a is chosen as the initial weighted variance. This does not alter the drift,
# but it keeps the exact requested form and makes the centering explicit.
a = weighted_variance(initial_positions, masses)

sim = simulate_variance_htransform_pair(
    initial_positions=initial_positions,
    masses=masses,
    lambda_=lambda_,
    a=a,
    horizon=horizon,
    step_size=step_size,
    seed=seed,
)

print(f"n_particles = {n_particles}")
print(f"lambda = {lambda_:.1f}")
print(f"a = initial weighted variance = {a:.6f}")
print(f"horizon = {horizon:.6e}, step_size = {step_size:.6e}, seed = {seed}")
print()
print(f"initial variance           = {sim.variances_free[0]:.6f}")
print(f"terminal variance (free)   = {sim.variances_free[-1]:.6f}")
print(f"terminal variance (biased) = {sim.variances_conditioned[-1]:.6f}")
print(f"variance reduction factor  = {sim.variances_free[-1] / sim.variances_conditioned[-1]:.3f}x")
print(f"max barycenter gap under shared noise = {np.max(np.linalg.norm(sim.weighted_means_free - sim.weighted_means_conditioned, axis=1)):.3e}")


n_particles = 128
lambda = 20000.0
a = initial weighted variance = 0.031292
horizon = 5.000000e-05, step_size = 2.083333e-07, seed = 4

initial variance           = 0.031292
terminal variance (free)   = 0.060006
terminal variance (biased) = 0.006845
variance reduction factor  = 8.766x
max barycenter gap under shared noise = 6.684e-16


In [5]:

snapshot_indices = [
    0,
    len(sim.times) // 3,
    2 * len(sim.times) // 3,
    len(sim.times) - 1,
]

snapshot_fig = make_snapshot_grid(
    sim=sim,
    color_values=blob_id,
    snapshot_indices=snapshot_indices,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
)
snapshot_fig.show()


In [6]:

animation_fig = make_comparison_animation(
    sim=sim,
    color_values=blob_id,
    blob_centers=blob_centers,
    blob_radius=blob_radius,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
    title="Two-blob collapse induced by the variance h-transform",
)
animation_fig.show()


In [7]:

variance_fig = make_variance_path_figure(sim)
variance_fig.show()



## Robustness across seeds

The next cell repeats the same many-particle experiment for several seeds.
The point is not just that the biased process has smaller variance on average;
the point is that the visual effect remains strong **seed after seed**.


In [8]:

seed_df = seed_sweep_dataframe(
    n_particles=128,
    lambda_=lambda_,
    horizon=horizon,
    step_size=step_size,
    seeds=range(10),
)
display(seed_df)

seed_fig = go.Figure()
seed_fig.add_trace(
    go.Scatter(
        x=seed_df["seed"],
        y=seed_df["terminal_variance_free"],
        mode="lines+markers",
        name="free diffusion",
    )
)
seed_fig.add_trace(
    go.Scatter(
        x=seed_df["seed"],
        y=seed_df["terminal_variance_conditioned"],
        mode="lines+markers",
        name="variance-conditioned h-transform",
    )
)
seed_fig.update_layout(
    title="Terminal variance over 10 seeds (n = 128)",
    template="simple_white",
    width=860,
    height=420,
    xaxis_title="seed",
    yaxis_title="terminal weighted variance",
)
seed_fig.show()

print(f"worst-case reduction factor over these seeds = {seed_df['reduction_factor'].min():.3f}x")
print(f"mean reduction factor over these seeds       = {seed_df['reduction_factor'].mean():.3f}x")


,seed,terminal_variance_free,terminal_variance_conditioned,reduction_factor
0,0,0.051692,0.005811,8.894995
1,1,0.057213,0.006255,9.146483
2,2,0.048575,0.004999,9.717122
3,3,0.054935,0.006646,8.265830
4,4,0.060006,0.006845,8.766433
5,5,0.062902,0.006855,9.176084
6,6,0.053223,0.005941,8.958897
7,7,0.057147,0.006079,9.401392
8,8,0.055058,0.006592,8.352482
9,9,0.050893,0.005341,9.529477


worst-case reduction factor over these seeds = 8.266x
mean reduction factor over these seeds       = 9.021x


## Size sweep above 100 particles

The main animation already uses $n = 128$, but the next check verifies that the effect persists for several values with $n \ge 100$.


In [9]:

size_df = size_sweep_dataframe(
    n_values=[100, 128, 160],
    lambda_=lambda_,
    horizon=horizon,
    step_size=step_size,
    seeds=range(6),
)
display(size_df)


,n_particles,mean_terminal_variance_free,mean_terminal_variance_conditioned,worst_case_reduction_factor
0,100,0.050132,0.005208,8.899881
1,128,0.055887,0.006235,8.265830
2,160,0.063566,0.007776,7.959375


## Takeaway

This notebook gives an example that is qualitatively very different from the labelled Gaussian target:

- the terminal weight depends only on a global statistic of the measure
- the exact heat solve produces a collective contraction drift
- no endpoint is prescribed for any individual particle
- the effect is already very strong with 128 particles, and it stays strong across different seeds

If you later want $a$ itself to change the drift, rather than only rescale the martingale density, the natural next experiment would be

$$
g(\mu) = \exp\bigl(-\lambda(\operatorname{Var}(\mu)-a)^2\bigr).
$$

I have not used that here because this notebook sticks to the exact linear penalty

$$
g(\mu) = \exp\bigl(-\lambda(\operatorname{Var}(\mu)-a)\bigr).
$$
